# 2단계: 앙상블 (선택)

| 순서 | 노트북 | 내용 |
|---|---|---|
| 0 | `ml_0_dataset` | 학습 데이터셋 결정: 모델 4종 × 변수 묶음 4종을 705명 / 원래 445명 출신으로 각각 학습해 비교 |
| 1 | `ml_1_main` | 0에서 정한 데이터셋의 결과로 모델 비교 (0에서 학습한 결과를 불러옴) |
| 2 | `ml_2_ensemble` | 앙상블 (선택, 최종 모델을 고르기 **전에**) |
| 3 | `ml_1_main` 6번 | **최종 모델 1개** 선택 |
| 4 | `ml_3_final_test` | 최종 모델 학습과 test 1회 평가 |

최종 모델을 고르기 **전에** 실행합니다. 앙상블도 최종 모델 후보 중 하나이기 때문입니다.

1단계에서 학습한 모델들의 OOF 예측을 합칩니다. **새로 학습하지 않고**, 모두 같은 fold에서 만든 예측이라 바로 합칠 수 있습니다.

| 방식 | 내용 | 용도 |
|---|---|---|
| `mean` | 확률 평균 | 최종 모델 후보가 될 수 있음 |
| `rank` | 반복마다 순위(0~1)를 매겨 평균 | 확률이 아니라 점수. 모델마다 확률 크기가 달라도 공평하게 섞이는지 보는 비교용 |

## 0. 준비

In [ ]:
import sys
from pathlib import Path

# 저장소 루트를 찾아 import 경로에 추가합니다 (노트북을 어느 폴더에서 열어도 동작).
repo = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Members/mingyu/model/ML/kfold").is_dir()
)
sys.path.insert(0, str(repo))

from Members.mingyu.model.ML.kfold import (  # noqa: E402
    Method,
    Settings,
    compare,
    load_cohort,
    make_ensemble,
    save_or_check_folds,
)  # noqa: E402

settings = Settings()
cohort = load_cohort(settings)
save_or_check_folds(cohort, settings)  # 처음엔 fold를 저장하고, 이후엔 같은 fold인지 확인

print(
    f"불러온 개발 데이터 {len(cohort.ids)}명 | 이 노트북에서 쓰는 환자: 원래 445명 출신 {cohort.original.sum()}명 "
    f"(SCD {cohort.y[cohort.original].sum()}) | test {len(cohort.test_ids)}명"
)
print("결과 폴더:", settings.result_dir)

## 1. 합칠 모델

`TRAIN_FILTER`는 `ml_1_main`과 같게 둡니다. `ml_1_main` 결과를 보고 성격이 다른 상위 모델(예: 로지스틱 + 트리)을 고릅니다.

In [ ]:
TRAIN_FILTER = "only445"  # ml_0_dataset 결정 (ml_1_main과 같게)
POPULATION = "all" if TRAIN_FILTER == "all" else "original"

reference = Method("Logistic", "ProBNP", TRAIN_FILTER).name
members = [
    # Method("Logistic", "MUSIC_SCD", TRAIN_FILTER).name,
    # Method("XGBoost", "core", TRAIN_FILTER).name,
]
assert len(members) >= 2, "ml_1_main 결과를 보고 합칠 모델을 2개 이상 적어 주세요"

## 2. 확률 평균 앙상블

In [ ]:
ensemble_mean = make_ensemble(settings, cohort, members, "mean")

## 3. 순위 평균 앙상블 (비교용)

In [ ]:
ensemble_rank = make_ensemble(settings, cohort, members, "rank")

## 4. 비교

순위 평균은 확률이 아니라서 Brier, log loss, 민감도/특이도 칸이 비어 있습니다.

In [ ]:
compare(
    settings,
    cohort,
    [reference, *members, ensemble_mean, ensemble_rank],
    reference=reference,
    population=POPULATION,
    save_as="ensemble",
)